# This notebook will demonstrate how to filter wikipedia for science articles

The science articles can later be used to generate questions and answers.  

I do this in 4 steps

1. Download wikipedia dataset
2. Label a few random samples using Gradio. Use given dataset as positive labels
3. Train a classifier on those labeled samples
4. Use the classifier to label the rest of the dataset


I am using the 2022-03-01 snapshot that is on the [Hugging Face Hub](https://huggingface.co/datasets/wikipedia). I have also uploaded it to Kaggle [here](https://www.kaggle.com/datasets/nbroad/wiki-20220301-en). It has 6.5M rows.
The final science dataset has been uploaded [here](https://www.kaggle.com/datasets/nbroad/wiki-20220301-en-sci). It has 131k examples

The science/not science classifier is on my Hugging Face profile: https://huggingface.co/nbroad/sciwiki-e5-sm


Note: The labels were what I considered to be relevant to STEM. You can follow this process with your own labels if you have a different definition of what is considered STEM.


In [1]:
!pip install -U gradio setfit datasets accelerate transformers -q

# Step 1: Streaming the wikipedia dataset

I stream this dataset from the Hugging Face Hub. If fully downloading, you'll need apache_beam and 40GB of space. By streaming, you won't have to store anything. There is only one snapshot preprocessed on the HF Hub. If you want more, you'll need to follow the [instructions on the page.](https://huggingface.co/datasets/wikipedia)

This snapshot has 6.5M rows, and I'll pull 200 examples by skipping around to make sure I get a diverse set.

Alternatively, you can use the dataset I uploaded to Kaggle [here](https://www.kaggle.com/datasets/nbroad/wiki-20220301-en). Even though the files are 11GB, using the Hugging Face `datasets` library means that it is memory mapped and will use [very little RAM (~50MB)](https://huggingface.co/docs/datasets/v2.13.1/en/about_arrow#memorymapping).

In [2]:
from datasets import load_dataset
from tqdm.auto import tqdm

ds = load_dataset("wikipedia", "20220301.en", streaming=True, split="train")


total_number_of_samples = 6458670 # https://huggingface.co/datasets/wikipedia#data-splits

chunk_size = 10
skip_size = 5000
num_repeat = 20

assert num_repeat*(skip_size+chunk_size) < total_number_of_samples

it_ds = iter(ds)

data = []
for _ in tqdm(range(num_repeat)):
    data.extend([next(it_ds)["text"] for _ in range(chunk_size)])
    # skip some
    [next(it_ds) for _ in range(skip_size)]

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


  0%|          | 0/20 [00:00<?, ?it/s]

In [3]:
# Save samples for later

import json

with open('wiki_samples.json', "w") as fp:
    json.dump(data, fp)

In [4]:
# create this in its own cell so it doesn't get overwritten
idx2label = {}

# Step 2: Create gradio app to label quickly

It took less than 5 minutes to label 200 examples

The Kaggle output notebook doesn't show the app. It is supposed to look like this: ![](https://raw.githubusercontent.com/nbroad1881/kaggle-images/main/llm-science-exam/create-science-dataset.gif)

In [ ]:
import gradio as gr

def get_text(idx):
    """
    Show first 500 characters at index `idx`
    """
    global data
    
    return data[idx][:500]

def update_idx(idx):
    
    return get_text(idx), idx2label.get(idx, "unlabeled")

def label_as_science(idx):
    
    global idx2label
    
    idx2label[idx] = 1
    
    new_idx = idx + 1
    
    return new_idx, get_text(new_idx), idx2label.get(new_idx, "unlabeled")
    
def label_as_not_science(idx):
    
    global idx2label
        
    idx2label[idx] = 0
    
    new_idx = idx + 1
    
    return new_idx, get_text(new_idx), idx2label.get(new_idx, "unlabeled")

def prev_fn(idx):
    global idx2label
    return idx-1, get_text(idx-1), idx2label.get(idx-1, "unlabeled")

def next_fn(idx):
    global idx2label
    return idx+1, get_text(idx+1), idx2label.get(idx+1, "unlabeled")

with gr.Blocks() as demo:
    
    starting_idx = 0
    
    with gr.Row():
        with gr.Column():
            idx = gr.Slider(value=starting_idx, label="idx")
            text = gr.Textbox(value=get_text(0), label="text")

        with gr.Column():
            
            with gr.Row():
                sci_btn = gr.Button(value="Science")
                not_sci_btn = gr.Button(value="NOT Science")
            
            with gr.Row():
                previous = gr.Button(value="Previous example")
                next_ = gr.Button(value="Next example")
                label = gr.Textbox(value=idx2label.get(0, "unlabeled"), label="Label")
    
    previous.click(fn=prev_fn, inputs=idx, outputs=[idx, text, label])
    next_.click(fn=next_fn, inputs=idx, outputs=[idx, text, label])
    idx.change(fn=update_idx, inputs=idx, outputs=[text, label])
    sci_btn.click(fn=label_as_science, inputs=[idx], outputs=[idx, text, label])
    not_sci_btn.click(fn=label_as_not_science, inputs=[idx], outputs=[idx, text, label])
    
    
demo.launch(debug=True, show_error=True)

Running on local URL:  http://127.0.0.1:7860
Kaggle notebooks require sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Running on public URL: https://990d6420396aca10fa.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


### It is heavily skewed toward non-science articles

This is fine because we were given 200 science texts from the hosts

In [8]:
import json
from pathlib import Path
from collections import Counter


f = Path("idx2label.json")

if f.exists():
    print("*"*50)
    print("Are you sure you want to overwrite existing file? Comment this out to overwrite")
    print("*"*50)

else:
    with open("idx2label.json", "w") as fp:
        json.dump(idx2label, fp)
    
with open("idx2label.json", "r") as fp:
    idx2label = json.load(fp)
    

Counter([x for x in idx2label.values()])

**************************************************
Are you sure you want to overwrite existing file? Comment this out to overwrite
**************************************************


Counter({0: 190, 1: 10})

# Step 3: Training the model using setfit

[setfit](https://github.com/huggingface/setfit) is an easy way to make a classifier on a small amount of data. 

![](https://raw.githubusercontent.com/huggingface/setfit/main/assets/setfit.png)

I use 2x T4 because it can use AMP, but unfortunately setfit can only train on a single GPU, so one of the T4 will be idle.

In [6]:
%%writefile train_setfit.py

import json
import os
from datasets import Dataset
import pandas as pd
from sentence_transformers.losses import CosineSimilarityLoss

from setfit import SetFitTrainer, SetFitModel

# Can only train on one gpu
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# a good sentence embedding model
model_id = "intfloat/e5-small-v2"
model = SetFitModel.from_pretrained(model_id)


# All of the examples in train.csv from Kaggle will get the label "science"
df = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/train.csv")
train_texts = []
options = list("ABCDE")
for *cols, prompt, answer in df[list("ABCDE") + ["prompt", "answer"]].values:
    train_texts.append(prompt + "\n" + cols[options.index(answer)])
    

with open("/kaggle/working/wiki_samples.json") as fp:
    wiki_samples = json.load(fp)

with open("/kaggle/working/idx2label.json") as fp:
    idx2label = json.load(fp)
    

idxs = list(idx2label.keys())

texts = [wiki_samples[int(i)] for i in idxs] + train_texts
labels = [idx2label[i] for i in idxs] + [1]*len(train_texts)

ds = Dataset.from_dict({"text": texts, "label": labels})
split_ds = ds.train_test_split(0.2)
split_ds["train"].to_json("train.json")
split_ds["test"].to_json("eval.json")

trainer = SetFitTrainer(
    model=model,
    train_dataset=split_ds["train"],
    eval_dataset=split_ds["test"],
    loss_class=CosineSimilarityLoss,
    num_iterations=10, # might get better results by training on more pairs, more epochs
    column_mapping={"text": "text", "label": "label"},
    batch_size=8,
    use_amp=True,
)

# progress bar looks terrible in kaggle
trainer.train(show_progress_bar=False)

model.save_pretrained("setfit-sciwiki")

print(trainer.evaluate())

Overwriting train_setfit.py


In [7]:
!python train_setfit.py

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

# Testing it out

tensor([[0.0237, 0.9763]]) means 0.9789 confidence it is science-related

In [8]:
from setfit import SetFitModel

model = SetFitModel.from_pretrained("setfit-sciwiki")

model.predict_proba(["General relativity, also known as the general theory of relativity \
and Einstein's theory of gravity, is the geometric theory of gravitation published \
by Albert Einstein in 1915 and is the current description of gravitation in modern physics."])

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.0237, 0.9763]], dtype=torch.float64)

In [10]:
model.predict_proba(["The potato is a starchy food, a tuber of the plant Solanum \
tuberosum and is a root vegetable native to the Americas. The plant \
is a perennial in the nightshade family Solanaceae. Wild potato species\
can be found from the southern United States to southern Chile."])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.0260, 0.9740]], dtype=torch.float64)

In [11]:
model.predict_proba(["The United States of America (U.S.A. or USA), commonly known \
as the United States (U.S. or US) or America, is a country, primarily \
located in North America, that consists of 50 states, a federal district, \
five major unincorporated territories, nine Minor Outlying Islands,[i] and \
326 Indian reservations. It is the world's third-largest country by both land \
and total area.[c] It shares land borders with Canada to its north and with \
Mexico to its south and has maritime borders with the Bahamas, Cuba, Russia, and other nations."])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.9655, 0.0345]], dtype=torch.float64)

In [12]:
model.predict_proba(["Sacramento, capital of the U.S. state of California, lies at the confluence \
of the Sacramento River and American River. The district of Old Sacramento harkens \
back to the city’s Gold Rush era, with wooden sidewalks and wagon rides. One of \
several museums in Old Sacramento, the California State Railroad Museum depicts the \
construction of the Transcontinental Railroad, one of the country’s earliest technological feats."])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.8325, 0.1675]], dtype=torch.float64)

In [13]:
model.predict_proba(["Katrina Kaif (pronounced [kəˈʈriːna kɛːf]; born Katrina Turquotte; 16 July 1983)[a] \
is a British actress who works in Hindi-language films. One of the highest-paid actresses in India, \
she has received accolades, including four Screen Awards and four Zee Cine Awards, in addition to three\
Filmfare nominations. Though reception to her acting has varied, she is noted for her dancing ability in \
various successful item numbers. "])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.9685, 0.0315]], dtype=torch.float64)

In [14]:
model.predict_proba(["""The SN2 reaction is a type of reaction mechanism that is common in \
organic chemistry. In this mechanism, one bond is broken and one bond is formed\
in a concerted way, i.e., in one step. The name SN2 refers to the Hughes-Ingold \
symbol of the mechanism: "SN" indicates that the reaction is a nucleophilic substitution, \
and "2" that it proceeds via a bi-molecular mechanism, which means both the reacting \
species are involved in the rate-determining step. The other major type of nucleophilic \
substitution is the SN1, but many other more specialized mechanisms describe substitution reactions. """])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.0333, 0.9667]], dtype=torch.float64)

# Step 3': Using a regular fine-tuning approach

After using setfit and getting slow inference speeds, I realized that it would take an enormous amount of time to run through the whole dataset. I used the run_glue.py example script to quickly fine-tune a model and it runs ~100x times faster. 

In [15]:
%%writefile run_glue.py

#!/usr/bin/env python
# coding=utf-8
# Copyright 2020 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
""" Finetuning the library models for sequence classification on GLUE."""
# You can also adapt this script on your own text classification task. Pointers for this are left as comments.

import logging
import os
import random
import sys
from dataclasses import dataclass, field
from typing import Optional

import datasets
import evaluate
import numpy as np
from datasets import load_dataset

import transformers
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    HfArgumentParser,
    PretrainedConfig,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import check_min_version, send_example_telemetry
from transformers.utils.versions import require_version

os.environ["TOKENIZERS_PARALLELISM"] = "false"

require_version("datasets>=1.8.0", "To fix: pip install -r examples/pytorch/text-classification/requirements.txt")

task_to_keys = {
    "cola": ("sentence", None),
    "mnli": ("premise", "hypothesis"),
    "mrpc": ("sentence1", "sentence2"),
    "qnli": ("question", "sentence"),
    "qqp": ("question1", "question2"),
    "rte": ("sentence1", "sentence2"),
    "sst2": ("sentence", None),
    "stsb": ("sentence1", "sentence2"),
    "wnli": ("sentence1", "sentence2"),
}

logger = logging.getLogger(__name__)


@dataclass
class DataTrainingArguments:
    """
    Arguments pertaining to what data we are going to input our model for training and eval.

    Using `HfArgumentParser` we can turn this class
    into argparse arguments to be able to specify them on
    the command line.
    """

    task_name: Optional[str] = field(
        default=None,
        metadata={"help": "The name of the task to train on: " + ", ".join(task_to_keys.keys())},
    )
    dataset_name: Optional[str] = field(
        default=None, metadata={"help": "The name of the dataset to use (via the datasets library)."}
    )
    dataset_config_name: Optional[str] = field(
        default=None, metadata={"help": "The configuration name of the dataset to use (via the datasets library)."}
    )
    max_seq_length: int = field(
        default=128,
        metadata={
            "help": (
                "The maximum total input sequence length after tokenization. Sequences longer "
                "than this will be truncated, sequences shorter will be padded."
            )
        },
    )
    overwrite_cache: bool = field(
        default=False, metadata={"help": "Overwrite the cached preprocessed datasets or not."}
    )
    pad_to_max_length: bool = field(
        default=True,
        metadata={
            "help": (
                "Whether to pad all samples to `max_seq_length`. "
                "If False, will pad the samples dynamically when batching to the maximum length in the batch."
            )
        },
    )
    max_train_samples: Optional[int] = field(
        default=None,
        metadata={
            "help": (
                "For debugging purposes or quicker training, truncate the number of training examples to this "
                "value if set."
            )
        },
    )
    max_eval_samples: Optional[int] = field(
        default=None,
        metadata={
            "help": (
                "For debugging purposes or quicker training, truncate the number of evaluation examples to this "
                "value if set."
            )
        },
    )
    max_predict_samples: Optional[int] = field(
        default=None,
        metadata={
            "help": (
                "For debugging purposes or quicker training, truncate the number of prediction examples to this "
                "value if set."
            )
        },
    )
    train_file: Optional[str] = field(
        default=None, metadata={"help": "A csv or a json file containing the training data."}
    )
    validation_file: Optional[str] = field(
        default=None, metadata={"help": "A csv or a json file containing the validation data."}
    )
    test_file: Optional[str] = field(default=None, metadata={"help": "A csv or a json file containing the test data."})

    def __post_init__(self):
        if self.task_name is not None:
            self.task_name = self.task_name.lower()
            if self.task_name not in task_to_keys.keys():
                raise ValueError("Unknown task, you should pick one in " + ",".join(task_to_keys.keys()))
        elif self.dataset_name is not None:
            pass
        elif self.train_file is None or self.validation_file is None:
            raise ValueError("Need either a GLUE task, a training/validation file or a dataset name.")
        else:
            train_extension = self.train_file.split(".")[-1]
            assert train_extension in ["csv", "json"], "`train_file` should be a csv or a json file."
            validation_extension = self.validation_file.split(".")[-1]
            assert (
                validation_extension == train_extension
            ), "`validation_file` should have the same extension (csv or json) as `train_file`."


@dataclass
class ModelArguments:
    """
    Arguments pertaining to which model/config/tokenizer we are going to fine-tune from.
    """

    model_name_or_path: str = field(
        metadata={"help": "Path to pretrained model or model identifier from huggingface.co/models"}
    )
    config_name: Optional[str] = field(
        default=None, metadata={"help": "Pretrained config name or path if not the same as model_name"}
    )
    tokenizer_name: Optional[str] = field(
        default=None, metadata={"help": "Pretrained tokenizer name or path if not the same as model_name"}
    )
    cache_dir: Optional[str] = field(
        default=None,
        metadata={"help": "Where do you want to store the pretrained models downloaded from huggingface.co"},
    )
    use_fast_tokenizer: bool = field(
        default=True,
        metadata={"help": "Whether to use one of the fast tokenizer (backed by the tokenizers library) or not."},
    )
    model_revision: str = field(
        default="main",
        metadata={"help": "The specific model version to use (can be a branch name, tag name or commit id)."},
    )
    use_auth_token: bool = field(
        default=False,
        metadata={
            "help": (
                "Will use the token generated when running `huggingface-cli login` (necessary to use this script "
                "with private models)."
            )
        },
    )
    ignore_mismatched_sizes: bool = field(
        default=False,
        metadata={"help": "Will enable to load a pretrained model whose head dimensions are different."},
    )


def main():
    # See all possible arguments in src/transformers/training_args.py
    # or by passing the --help flag to this script.
    # We now keep distinct sets of args, for a cleaner separation of concerns.

    parser = HfArgumentParser((ModelArguments, DataTrainingArguments, TrainingArguments))
    if len(sys.argv) == 2 and sys.argv[1].endswith(".json"):
        # If we pass only one argument to the script and it's the path to a json file,
        # let's parse it to get our arguments.
        model_args, data_args, training_args = parser.parse_json_file(json_file=os.path.abspath(sys.argv[1]))
    else:
        model_args, data_args, training_args = parser.parse_args_into_dataclasses()

    # Sending telemetry. Tracking the example usage helps us better allocate resources to maintain them. The
    # information sent is the one passed as arguments along with your Python/PyTorch versions.
    send_example_telemetry("run_glue", model_args, data_args)

    # Setup logging
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        handlers=[logging.StreamHandler(sys.stdout)],
    )

    if training_args.should_log:
        # The default of training_args.log_level is passive, so we set log level at info here to have that default.
        transformers.utils.logging.set_verbosity_info()

    log_level = training_args.get_process_log_level()
    logger.setLevel(log_level)
    datasets.utils.logging.set_verbosity(log_level)
    transformers.utils.logging.set_verbosity(log_level)
    transformers.utils.logging.enable_default_handler()
    transformers.utils.logging.enable_explicit_format()

    # Log on each process the small summary:
    logger.warning(
        f"Process rank: {training_args.local_rank}, device: {training_args.device}, n_gpu: {training_args.n_gpu}"
        + f"distributed training: {bool(training_args.local_rank != -1)}, 16-bits training: {training_args.fp16}"
    )
    logger.info(f"Training/evaluation parameters {training_args}")

    # Detecting last checkpoint.
    last_checkpoint = None
    if os.path.isdir(training_args.output_dir) and training_args.do_train and not training_args.overwrite_output_dir:
        last_checkpoint = get_last_checkpoint(training_args.output_dir)
        if last_checkpoint is None and len(os.listdir(training_args.output_dir)) > 0:
            raise ValueError(
                f"Output directory ({training_args.output_dir}) already exists and is not empty. "
                "Use --overwrite_output_dir to overcome."
            )
        elif last_checkpoint is not None and training_args.resume_from_checkpoint is None:
            logger.info(
                f"Checkpoint detected, resuming training at {last_checkpoint}. To avoid this behavior, change "
                "the `--output_dir` or add `--overwrite_output_dir` to train from scratch."
            )

    # Set seed before initializing model.
    set_seed(training_args.seed)

    # Get the datasets: you can either provide your own CSV/JSON training and evaluation files (see below)
    # or specify a GLUE benchmark task (the dataset will be downloaded automatically from the datasets Hub).
    #
    # For CSV/JSON files, this script will use as labels the column called 'label' and as pair of sentences the
    # sentences in columns called 'sentence1' and 'sentence2' if such column exists or the first two columns not named
    # label if at least two columns are provided.
    #
    # If the CSVs/JSONs contain only one non-label column, the script does single sentence classification on this
    # single column. You can easily tweak this behavior (see below)
    #
    # In distributed training, the load_dataset function guarantee that only one local process can concurrently
    # download the dataset.
    if data_args.task_name is not None:
        # Downloading and loading a dataset from the hub.
        raw_datasets = load_dataset(
            "glue",
            data_args.task_name,
            cache_dir=model_args.cache_dir,
            use_auth_token=True if model_args.use_auth_token else None,
        )
    elif data_args.dataset_name is not None:
        # Downloading and loading a dataset from the hub.
        raw_datasets = load_dataset(
            data_args.dataset_name,
            data_args.dataset_config_name,
            cache_dir=model_args.cache_dir,
            use_auth_token=True if model_args.use_auth_token else None,
        )
    else:
        # Loading a dataset from your local files.
        # CSV/JSON training and evaluation files are needed.
        data_files = {"train": data_args.train_file, "validation": data_args.validation_file}

        # Get the test dataset: you can provide your own CSV/JSON test file (see below)
        # when you use `do_predict` without specifying a GLUE benchmark task.
        if training_args.do_predict:
            if data_args.test_file is not None:
                train_extension = data_args.train_file.split(".")[-1]
                test_extension = data_args.test_file.split(".")[-1]
                assert (
                    test_extension == train_extension
                ), "`test_file` should have the same extension (csv or json) as `train_file`."
                data_files["test"] = data_args.test_file
            else:
                raise ValueError("Need either a GLUE task or a test file for `do_predict`.")

        for key in data_files.keys():
            logger.info(f"load a local file for {key}: {data_files[key]}")

        if data_args.train_file.endswith(".csv"):
            # Loading a dataset from local csv files
            raw_datasets = load_dataset(
                "csv",
                data_files=data_files,
                cache_dir=model_args.cache_dir,
                use_auth_token=True if model_args.use_auth_token else None,
            )
        else:
            # Loading a dataset from local json files
            raw_datasets = load_dataset(
                "json",
                data_files=data_files,
                cache_dir=model_args.cache_dir,
                use_auth_token=True if model_args.use_auth_token else None,
            )
    # See more about loading any type of standard or custom dataset at
    # https://huggingface.co/docs/datasets/loading_datasets.html.

    # Labels
    if data_args.task_name is not None:
        is_regression = data_args.task_name == "stsb"
        if not is_regression:
            label_list = raw_datasets["train"].features["label"].names
            num_labels = len(label_list)
        else:
            num_labels = 1
    else:
        # Trying to have good defaults here, don't hesitate to tweak to your needs.
        is_regression = raw_datasets["train"].features["label"].dtype in ["float32", "float64"]
        if is_regression:
            num_labels = 1
        else:
            # A useful fast method:
            # https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasets.Dataset.unique
            label_list = raw_datasets["train"].unique("label")
            label_list.sort()  # Let's sort it for determinism
            num_labels = len(label_list)

    # Load pretrained model and tokenizer
    #
    # In distributed training, the .from_pretrained methods guarantee that only one local process can concurrently
    # download model & vocab.
    config = AutoConfig.from_pretrained(
        model_args.config_name if model_args.config_name else model_args.model_name_or_path,
        num_labels=num_labels,
        finetuning_task=data_args.task_name,
        cache_dir=model_args.cache_dir,
        revision=model_args.model_revision,
        use_auth_token=True if model_args.use_auth_token else None,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_args.tokenizer_name if model_args.tokenizer_name else model_args.model_name_or_path,
        cache_dir=model_args.cache_dir,
        use_fast=model_args.use_fast_tokenizer,
        revision=model_args.model_revision,
        use_auth_token=True if model_args.use_auth_token else None,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_args.model_name_or_path,
        from_tf=bool(".ckpt" in model_args.model_name_or_path),
        config=config,
        cache_dir=model_args.cache_dir,
        revision=model_args.model_revision,
        use_auth_token=True if model_args.use_auth_token else None,
        ignore_mismatched_sizes=model_args.ignore_mismatched_sizes,
    )

    # Preprocessing the raw_datasets
    if data_args.task_name is not None:
        sentence1_key, sentence2_key = task_to_keys[data_args.task_name]
    else:
        # Again, we try to have some nice defaults but don't hesitate to tweak to your use case.
        non_label_column_names = [name for name in raw_datasets["train"].column_names if name != "label"]
        if "sentence1" in non_label_column_names and "sentence2" in non_label_column_names:
            sentence1_key, sentence2_key = "sentence1", "sentence2"
        else:
            if len(non_label_column_names) >= 2:
                sentence1_key, sentence2_key = non_label_column_names[:2]
            else:
                sentence1_key, sentence2_key = non_label_column_names[0], None

    # Padding strategy
    if data_args.pad_to_max_length:
        padding = "max_length"
    else:
        # We will pad later, dynamically at batch creation, to the max sequence length in each batch
        padding = False

    # Some models have set the order of the labels to use, so let's make sure we do use it.
    label_to_id = None
    if (
        model.config.label2id != PretrainedConfig(num_labels=num_labels).label2id
        and data_args.task_name is not None
        and not is_regression
    ):
        # Some have all caps in their config, some don't.
        label_name_to_id = {k.lower(): v for k, v in model.config.label2id.items()}
        if sorted(label_name_to_id.keys()) == sorted(label_list):
            label_to_id = {i: int(label_name_to_id[label_list[i]]) for i in range(num_labels)}
        else:
            logger.warning(
                "Your model seems to have been trained with labels, but they don't match the dataset: ",
                f"model labels: {sorted(label_name_to_id.keys())}, dataset labels: {sorted(label_list)}."
                "\nIgnoring the model labels as a result.",
            )
    elif data_args.task_name is None and not is_regression:
        label_to_id = {v: i for i, v in enumerate(label_list)}

    if label_to_id is not None:
        model.config.label2id = label_to_id
        model.config.id2label = {id: label for label, id in config.label2id.items()}
    elif data_args.task_name is not None and not is_regression:
        model.config.label2id = {l: i for i, l in enumerate(label_list)}
        model.config.id2label = {id: label for label, id in config.label2id.items()}

    if data_args.max_seq_length > tokenizer.model_max_length:
        logger.warning(
            f"The max_seq_length passed ({data_args.max_seq_length}) is larger than the maximum length for the"
            f"model ({tokenizer.model_max_length}). Using max_seq_length={tokenizer.model_max_length}."
        )
    max_seq_length = min(data_args.max_seq_length, tokenizer.model_max_length)

    def preprocess_function(examples):
        # Tokenize the texts
        args = (
            (examples[sentence1_key],) if sentence2_key is None else (examples[sentence1_key], examples[sentence2_key])
        )
        result = tokenizer(*args, padding=padding, max_length=max_seq_length, truncation=True)

        # Map labels to IDs (not necessary for GLUE tasks)
        if label_to_id is not None and "label" in examples:
            result["label"] = [(label_to_id[l] if l != -1 else -1) for l in examples["label"]]
        return result

    with training_args.main_process_first(desc="dataset map pre-processing"):
        raw_datasets = raw_datasets.map(
            preprocess_function,
            batched=True,
            load_from_cache_file=not data_args.overwrite_cache,
            desc="Running tokenizer on dataset",
        )
    if training_args.do_train:
        if "train" not in raw_datasets:
            raise ValueError("--do_train requires a train dataset")
        train_dataset = raw_datasets["train"]
        if data_args.max_train_samples is not None:
            max_train_samples = min(len(train_dataset), data_args.max_train_samples)
            train_dataset = train_dataset.select(range(max_train_samples))

    if training_args.do_eval:
        if "validation" not in raw_datasets and "validation_matched" not in raw_datasets:
            raise ValueError("--do_eval requires a validation dataset")
        eval_dataset = raw_datasets["validation_matched" if data_args.task_name == "mnli" else "validation"]
        if data_args.max_eval_samples is not None:
            max_eval_samples = min(len(eval_dataset), data_args.max_eval_samples)
            eval_dataset = eval_dataset.select(range(max_eval_samples))

    if training_args.do_predict or data_args.task_name is not None or data_args.test_file is not None:
        if "test" not in raw_datasets and "test_matched" not in raw_datasets:
            raise ValueError("--do_predict requires a test dataset")
        predict_dataset = raw_datasets["test_matched" if data_args.task_name == "mnli" else "test"]
        if data_args.max_predict_samples is not None:
            max_predict_samples = min(len(predict_dataset), data_args.max_predict_samples)
            predict_dataset = predict_dataset.select(range(max_predict_samples))

    # Log a few random samples from the training set:
    if training_args.do_train:
        for index in random.sample(range(len(train_dataset)), 3):
            logger.info(f"Sample {index} of the training set: {train_dataset[index]}.")

    # Get the metric function
    if data_args.task_name is not None:
        metric = evaluate.load("glue", data_args.task_name)
    elif is_regression:
        metric = evaluate.load("mse")
    else:
        metric = evaluate.load("accuracy")

    # You can define your custom compute_metrics function. It takes an `EvalPrediction` object (a namedtuple with a
    # predictions and label_ids field) and has to return a dictionary string to float.
    def compute_metrics(p: EvalPrediction):
        preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        preds = np.squeeze(preds) if is_regression else np.argmax(preds, axis=1)
        result = metric.compute(predictions=preds, references=p.label_ids)
        if len(result) > 1:
            result["combined_score"] = np.mean(list(result.values())).item()
        return result

    # Data collator will default to DataCollatorWithPadding when the tokenizer is passed to Trainer, so we change it if
    # we already did the padding.
    if data_args.pad_to_max_length:
        data_collator = default_data_collator
    elif training_args.fp16:
        data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
    else:
        data_collator = None

    # Initialize our Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset if training_args.do_train else None,
        eval_dataset=eval_dataset if training_args.do_eval else None,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    # Training
    if training_args.do_train:
        checkpoint = None
        if training_args.resume_from_checkpoint is not None:
            checkpoint = training_args.resume_from_checkpoint
        elif last_checkpoint is not None:
            checkpoint = last_checkpoint
        train_result = trainer.train(resume_from_checkpoint=checkpoint)
        metrics = train_result.metrics
        max_train_samples = (
            data_args.max_train_samples if data_args.max_train_samples is not None else len(train_dataset)
        )
        metrics["train_samples"] = min(max_train_samples, len(train_dataset))

        trainer.save_model()  # Saves the tokenizer too for easy upload

        trainer.log_metrics("train", metrics)
        trainer.save_metrics("train", metrics)
        trainer.save_state()

    # Evaluation
    if training_args.do_eval:
        logger.info("*** Evaluate ***")

        # Loop to handle MNLI double evaluation (matched, mis-matched)
        tasks = [data_args.task_name]
        eval_datasets = [eval_dataset]
        if data_args.task_name == "mnli":
            tasks.append("mnli-mm")
            valid_mm_dataset = raw_datasets["validation_mismatched"]
            if data_args.max_eval_samples is not None:
                max_eval_samples = min(len(valid_mm_dataset), data_args.max_eval_samples)
                valid_mm_dataset = valid_mm_dataset.select(range(max_eval_samples))
            eval_datasets.append(valid_mm_dataset)
            combined = {}

        for eval_dataset, task in zip(eval_datasets, tasks):
            metrics = trainer.evaluate(eval_dataset=eval_dataset)

            max_eval_samples = (
                data_args.max_eval_samples if data_args.max_eval_samples is not None else len(eval_dataset)
            )
            metrics["eval_samples"] = min(max_eval_samples, len(eval_dataset))

            if task == "mnli-mm":
                metrics = {k + "_mm": v for k, v in metrics.items()}
            if task is not None and "mnli" in task:
                combined.update(metrics)

            trainer.log_metrics("eval", metrics)
            trainer.save_metrics("eval", combined if task is not None and "mnli" in task else metrics)

    if training_args.do_predict:
        logger.info("*** Predict ***")

        # Loop to handle MNLI double evaluation (matched, mis-matched)
        tasks = [data_args.task_name]
        predict_datasets = [predict_dataset]
        if data_args.task_name == "mnli":
            tasks.append("mnli-mm")
            predict_datasets.append(raw_datasets["test_mismatched"])

        for predict_dataset, task in zip(predict_datasets, tasks):
            # Removing the `label` columns because it contains -1 and Trainer won't like that.
            predict_dataset = predict_dataset.remove_columns("label")
            predictions = trainer.predict(predict_dataset, metric_key_prefix="predict").predictions
            predictions = np.squeeze(predictions) if is_regression else np.argmax(predictions, axis=1)

            output_predict_file = os.path.join(training_args.output_dir, f"predict_results_{task}.txt")
            if trainer.is_world_process_zero():
                with open(output_predict_file, "w") as writer:
                    logger.info(f"***** Predict results {task} *****")
                    writer.write("index\tprediction\n")
                    for index, item in enumerate(predictions):
                        if is_regression:
                            writer.write(f"{index}\t{item:3.3f}\n")
                        else:
                            item = label_list[item]
                            writer.write(f"{index}\t{item}\n")

    kwargs = {"finetuned_from": model_args.model_name_or_path, "tasks": "text-classification"}
    if data_args.task_name is not None:
        kwargs["language"] = "en"
        kwargs["dataset_tags"] = "glue"
        kwargs["dataset_args"] = data_args.task_name
        kwargs["dataset"] = f"GLUE {data_args.task_name.upper()}"

    if training_args.push_to_hub:
        trainer.push_to_hub(**kwargs)
    else:
        trainer.create_model_card(**kwargs)


def _mp_fn(index):
    # For xla_spawn (TPUs)
    main()


if __name__ == "__main__":
    main()

Overwriting run_glue.py


In [18]:
!torchrun --nproc_per_node 2 run_glue.py \
  --model_name_or_path "intfloat/e5-small-v2" \
  --output_dir "e5-sm-finetuned" \
  --overwrite_output_dir \
  --per_device_train_batch_size 32 \
  --fp16 \
  --evaluation_strategy "epoch" \
  --save_strategy "epoch" \
  --save_total_limit 1 \
  --dataloader_num_workers 2 \
  --do_train \
  --do_eval \
  --train_file "train.json" \
  --validation_file "eval.json" \
  --report_to "none" \
  --log_level "warning"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_

# Step 4: Labeling the whole dataset

There are 6.5M, so this takes some time. I have the notebook set up to persist files so that if it crashes, nothing is lost.

In [41]:
%%writefile label.py

import os
import json
import argparse
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from accelerate import Accelerator
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_path', type=str, default=0)
    parser.add_argument("--num_pq_files", type=int, default=-1)
    args = parser.parse_args()
    
    output_dir = f"science_texts"
    os.makedirs(output_dir, exist_ok=True)
    
    accelerator = Accelerator()


    model = AutoModelForSequenceClassification.from_pretrained(args.model_path)
    tokenizer = AutoTokenizer.from_pretrained(args.model_path)

    batch_size = 64

    
    files = list(map(str, Path("/kaggle/input/wiki-20220301-en").glob("*.parquet")))
    
    if args.num_pq_files > 0:
        files = files[:args.num_pq_files]
    
    ds = load_dataset("parquet", data_files=files, split="train")

    def tokenize(examples, idxs):
        """
        Only need first 500 characters of text.
        Add length to sort and minimize padding
        """
        tokenized =  tokenizer(
            [x[:500] for x in examples["text"]], 
            padding=False, 
            truncation=True, 
            max_length=512,
            )

        tokenized["length"] = [len(t) for t in tokenized.input_ids]
        
        tokenized["idx"] = idxs

        return tokenized

    tds = ds.map(tokenize, batched=True, num_proc=4, with_indices=True)
    tds = tds.sort("length")
    
    # Ignore text with small number of tokens
    tds = tds.filter(lambda x: x["length"] > 100, num_proc=4)

    collator = DataCollatorWithPadding(tokenizer)

    def collate(examples):

        inputs = []
        for x in examples:
            inputs.append({"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]})

        batch = collator(inputs)

        batch["text"] = [x["text"] for x in examples]  
        batch["idx"] = torch.tensor([x["idx"] for x in examples])
        

        return batch
    

    dl = DataLoader(
        tds, 
        collate_fn=collate, 
        batch_size=batch_size, 
        shuffle=False, 
        drop_last=False, 
        num_workers=4, 
        pin_memory=True,
        )

    model, dl = accelerator.prepare(model, dl)


    idxs = []
    count = 0
    with torch.inference_mode():
        for i, batch in tqdm(enumerate(dl)):
            out = model(batch["input_ids"], batch["attention_mask"])
            
            logits = accelerator.gather_for_metrics(out.logits)
            
            batch_idxs = accelerator.gather_for_metrics(batch["idx"])
            
            
            if accelerator.is_main_process:
                
                for (_, pos_proba), idx in zip(logits.softmax(-1), batch_idxs.cpu().tolist()):
                    if pos_proba > 0.95:
                        idxs.append(idx)               
                
            
            # Save every 100
            if (i+1) % 200 == 0:
                filename = f"{output_dir}/batch{i}.pq"
                Dataset.from_dict({"text": ds.select(idxs)["text"]}).to_parquet(filename)
                idxs = []

    
    if len(idxs):
        filename = f"{output_dir}/batch{i}.pq"
        Dataset.from_dict({"text": ds.select(idxs)["text"]}).to_parquet(filename)

if __name__ == "__main__":
    main()

Overwriting label.py


### Note: I only run this on 250k samples instead of the whole dataset. I did run it on the whole thing outside of Kaggle and that is the dataset I uploaded.

Tokenization on Kaggle is so slow :( It would probably be better to do tokenization in a cpu notebook and then load that into a gpu notebook.

In [42]:
# a copy is on the HF hub: "nbroad/sciwiki-e5-sm"
# the one trained in this notebook is bad and I wouldn't recommend using it

!torchrun --nproc_per_node 2 label.py \
  --model_path "nbroad/sciwiki-e5-sm" \
  --num_pq_files 1 # set this to -1 to do all files

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_

# Let's look at what we created

The full dataset is here: https://www.kaggle.com/datasets/nbroad/wiki-20220301-en-sci

These examples are full of chemistry 

In [1]:
from pathlib import Path
from datasets import load_dataset

files = list(map(str, Path("/kaggle/working/science_texts").glob("*.pq")))
ds = iter(load_dataset("parquet", data_files=files, split="train").shuffle())

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Dataset parquet downloaded and prepared to /root/.cache/huggingface/datasets/parquet/default-af06992f5b075213/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901. Subsequent calls will reuse this data.


In [2]:
next(ds)["text"][:1000]

'The Mølmer–Sørensen gate is a two qubit gate used in trapped ion quantum computing. It was proposed by Klaus Mølmer and Anders Sørensen.  Their proposal also extends to gates on more than two qubits.\n\nImplementation \nTo implement the gate, two ions are irradiated with a bichromatic laser field with frequencies , where  is the energy splitting of the qubit states and  is a detuning close to the motional frequency of the ions. Depending on the interaction time,  this produces the states\n\nThe above can then be shown to produce a universal set of gates. The Mølmer–Sørensen gate has the advantage that it does not fail if the ions were not cooled completely to the ground state, and it does not require the ions to be individually addressed. However, this thermal insensitivity is only valid in the Lamb–Dicke regime, so most implementations first cool the ions to the motional ground state. An experiment was done by P.C. Haljan, K. A. Brickman, L. Deslauriers, P.J. Lee, and C. Monroe where

In [3]:
next(ds)["text"][:1000]

'Crohn’s disease (CD) of the vulva  is a rare extra intestinal condition, with granulomatous cutaneous lesions affecting the female genitalia. Lesions connected to the affected gut via a healthy tissue are referred to as metastatic lesions.\n\nPresentation\nMost common clinical manifestation of vulvar CD is unilateral labial swelling associated with chronic vulvar ulceration, along with perianal lesions which were reported in 48% of the patients.\n\nDermatologic inflammatory vulvo-vaginal lesions are usually caused by fistulas arising from the anus or rectum. However, not all inflammatory lesions within the genitalia are caused in fistulas fashion, even in patients suffering from gastrointestinal Crohn’s disease.\n\nSimilar clinical lesions have been reported in male genitalia affecting the penis and the scrotum.\n\nVulvar CD bears no typical symptoms and is only diagnosed and associated with gastrointestinal CD based on vulvar ulcers and hypertrophic lesions. Some patients do, however

In [4]:
next(ds)["text"][:1000]

'Madin-Darby canine kidney (MDCK) cells are a model mammalian cell line used in biomedical research. MDCK cells are used for a wide variety of cell biology studies including cell polarity, cell-cell adhesions (termed adherens junctions), collective cell motility, as well as responses to growth factors. It is one of few cell culture models that is suited for 3D cell culture and multicellular rearrangements known as branching morphogenesis.\n\nHistory \nFollowing the initial isolation in 1958 of epithelial cells from the kidney tubule of an adult Cocker Spaniel dog by Stewart H. Madin and Norman B. Darby, Jr., the cell line bearing their name was employed primarily as a model for viral infection of mammalian cells. Indeed, they chose to isolate kidney tubules with precisely this goal in mind, as they had previously succeeded with viral infection of cells derived from kidney tubules from other mammals. Thus the initial goal in isolating and culturing cells from this tissue was not to gene

In [5]:
next(ds)["text"][:1000]

'The Ryu–Takayanagi conjecture is a conjecture within holography that posits a quantitative relationship between the entanglement entropy of a conformal field theory and the geometry of an associated anti-de Sitter spacetime. The formula characterizes "holographic screens" in the bulk; that is, it specifies which regions of the bulk geometry are "responsible to particular information in the dual CFT". The conjecture is named after  and , who jointly published the result in 2006. As a result, the authors were awarded the 2015 New Horizons in Physics Prize for "fundamental ideas about entropy in quantum field theory and quantum gravity". The formula was generalized to a covariant form in 2007.\n\nMotivation\n\nThe thermodynamics of black holes suggests certain relationships between the entropy of black holes and their geometry. Specifically, the Bekenstein–Hawking area formula conjectures that the entropy of a black hole is proportional to its surface area:\n\nThe Bekenstein–Hawking entr

In [6]:
next(ds)["text"][:1000]

'The Caelum Supercluster, also known as SCl 59, may be a massive supercluster; spanning 910 million light-years, it is perhaps the largest galaxy supercluster in the universe. It has a mass of 2×1017 solar masses, 1.7 times the mass of Laniakea Supercluster and of Horologium Supercluster. It is centered on coordinates right ascension  and declination .\n\nThe nearest part of the supercluster is 1.4 billion light-years away from Earth, while the far end of it is 2.31 billion light-years, visible  in the constellations Caelum. The Caelum Supercluster has about 8,300 galaxy groups (50,000 giant galaxies and 500,000 dwarf galaxies).\n\nSee also\n Abell catalog\n Large scale structure of the universe\n List of Abell clusters\n List of superclusters\n List of largest galaxy superclusters\n\nReferences\n\nFurther reading\n\nGalaxy superclusters\nCaelum'

In [7]:
next(ds)["text"][:1000]

'Atribacterota is a phylum of bacteria, which are common in anoxic sediments rich in methane. They are distributed worldwide and in some cases abundant in anaerobic marine sediments, geothermal springs, and oil deposits. Genetic analyzes suggest a heterotrophic metabolism that gives rise to fermentation products such as acetate, ethanol, and . These products in turn can support methanogens within the sediment microbial community and explain the frequent occurrence of Atribacterota in methane-rich anoxic sediments. According to phylogenetic analysis, Atribacterota appears to be related to several thermophilic phyla within Terrabacteria or may be in the base of Gracilicutes. According to research, Atribacterota shows patterns of gene expressions which consists of fermentative, acetogenic metabolism. These expressions let Atribacterota to be able to create catabolic and anabolic functions which are necessary to generate cellular reproduction, even when the energy levels are limited due to

# Looking at random samples from the real dataset/kaggle/input/wiki-20220301-en-sci

In [8]:
files = list(map(str, Path("/kaggle/input/wiki-20220301-en-sci").glob("*.parquet")))
ds = iter(load_dataset("parquet", data_files=files, split="train").shuffle())

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Dataset parquet downloaded and prepared to /root/.cache/huggingface/datasets/parquet/default-098a509e5dd95ddd/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901. Subsequent calls will reuse this data.


In [9]:
next(ds)["text"][:1000]

'The concept of balancing derives from the balance of power theory, the most influential theory from the realist school of thought, which assumes that a formation of hegemony in a multistate system is unattainable since hegemony is perceived as a threat by other states, causing them to engage in balancing against a potential hegemon.\n\nBalancing encompasses the actions that a particular state or group of states take in order to equalise the odds against more powerful states; that is to make it more difficult and hence less likely for powerful states to exert their military advantage over the weaker ones.\n\nAccording to the balance of power theory, states, motivated primarily by their desire for survival and security, will develop and implement military capabilities and hard power mechanisms in order to constrain the most powerful and rising state that can prove a potential threat. This idea illustrates the concept of internal balancing, which is opposed to external, under which state

In [10]:
next(ds)["text"][:1000]

'1-Aminocyclopropane-1-carboxylic acid (ACC) is a disubstituted cyclic α-amino acid in which a cyclopropane ring is fused to the C atom of the amino acid.  It is a white solid.  Many cyclopropane-substituted amino acids are known, but this one occurs naturally. Like glycine, but unlike most α-amino acids, ACC is not chiral.\n\nBiochemistry\nACC is the precursor to  the plant hormone ethylene. It is synthesized by the enzyme ACC synthase ( ) from methionine and converted to ethylene by ACC oxidase ().\n\nACC also exhibits ethylene-independent signaling that plays a critical role in pollination and seed production by activating proteins similar to those involved in nervous system responses in humans and animals. More specifically, ACC signaling promotes secretion of the pollen tube chemoattractant LURE1.2 in ovular sporophytic tissue thus enhancing pollen tube attraction. Additionally, ACC activates Ca2+-containing ion currents via glutamate receptor-like (GLR) channels in root protoplas

In [11]:
next(ds)["text"][:1000]

"ξ Oph, Latinized as Xi Ophiuchi, is a visual binary star system in the equatorial constellation of Ophiuchus. It has a yellow-white hue and is faintly visible to the naked eye with a combined apparent visual magnitude of 4.39. The system is located approximately 56.6\xa0light years away from the Sun based on parallax, but is drifting closer with a radial velocity of -9\xa0km/s.\n\nThe magnitude 4.40 primary, designated component A, is an ordinary F-type main-sequence star with a stellar classification of F2V. It is 916\xa0million years old and is rotating with a projected rotational velocity of 20\xa0km/s. The star has 1.3 times the mass of the Sun and 1.6 times the Sun's radius. It is radiating 4.4 times the luminosity of the Sun from its photosphere at an effective temperature of 6,611\xa0K.\n\nThe system is a source of X-ray emission. The orbiting companion, component B, is a magnitude 8.9 star at an angular separation of  along a position angle of 27° from the primary, as of 2015.

In [12]:
next(ds)["text"][:1000]

"The Quadruplanar inversor of Sylvester and Kempe is a generalization of Hart's inversor. Like Hart's inversor, is a mechanism that provides a perfect straight line motion without sliding guides.\n\nThe mechanism was described in 1875 by James Joseph Sylvester in the journal Nature.\n\nLike Hart's inversor, it is based on an antiparallelogram (KBCD in the figures below) but the rather than placing the fixed, input and output points on the sides (dividing them in fixed proportion so they are all similar), Sylvester recognized that the additional points could be displaced sideways off the sides, as long as they formed similar triangles.  Hart's original form is simply the degenerate case of triangles with altitude zero.\n\nGallery \nIn these diagrams:\n KBCD is an antiparallelogram, so KB = CD and BC = DK.\n KBA, CBB0, and CDE (and KDI in the first diagram) are similar triangles.\n Because KB = CD, KBA and CDE are congruent triangles\n Because BC = DK, CBB0 and KDI are congruent triangle